# **Installation and Licensing**

In [ ]:
!pip install -q gamspy # Installs the GAMSPy package. GAMSPy documentation: https://gamspy.readthedocs.io/en/latest/

In [ ]:
!gamspy show license

In [ ]:
!gamspy list solvers

In [ ]:
!gamspy install license <path_to_ascii_file or access code> # paste your license here (if demo-license is not enough)

In [ ]:
# Optional: Install all solvers
!gamspy install solver --install-all-solvers

In [ ]:
!gamspy list solvers # Lists all available solvers in the gamspy package

# **Mathematical Model**

**Robust Portfolio Optimization with Cardinality and Transaction Costs**

\begin{align}
    \max \quad & t - \sum_{i=1}^n c_i (x_i - x_i^{prev})^2 \\
    \text{subject to} \quad
    & \sum_{i=1}^n x_i = 1 \quad && \text{(Budget)} \\
    & \sum_{i=1}^n y_i \leq 5 \quad && \text{(Cardinality)} \\
    & x_i \leq y_i, \quad \forall i \quad && \text{(Linking)} \\
    & \sum_{i=1}^n \sum_{j=1}^n Q_{ij} x_i x_j \leq 0.002 \quad && \text{(Risk)} \\
    & t + 0.005 \cdot z \leq \sum_{i=1}^n \mu_i x_i \quad && \text{(Robust return UB)} \\
    & \sum_{i=1}^n \sum_{j=1}^n \mu_i \mu_j x_i x_j \leq z \quad && \text{(SOC 1)} \\
    & -\sum_{i=1}^n \sum_{j=1}^n \mu_i \mu_j x_i x_j \leq z \quad && \text{(SOC 2)} \\
    & x_i \geq 0, \quad y_i \in \{0, 1\}, \quad \forall i \quad && \text{(Domains)}
\end{align}

$$
\begin{array}{l l l r r r r}
\text{Name} & \text{Type} & \text{C} & \#\text{Vars} & \#\text{BinVars} & \#\text{IntVars} & \#\text{Cons} \\
\hline
RPOPwCaTC5\_20 & MBQCQP & Convex & 42 & 20 & 0 & 26 \\
RPOPwCaTC5\_50 & MBQCQP & Convex & 102 & 50 & 0 & 56 \\
RPOPwCaTC5\_100 & MBQCQP & Convex & 202 & 100 & 0 & 106 \\
RPOPwCaTC5\_319 & MBQCQP & Convex & 640 & 319 & 0 & 325 \\
\end{array}
$$


#  **Solver Setup**


## RPOPwCaTC5\_20

In [ ]:
import yfinance as yf
import numpy as np
from gamspy import Container, Set, Parameter, Variable, Equation, Model, Sum, Sense, Alias

# 1. Load market data
tickers = [
    "BA", "CAT", "CVX", "DIS", "JNJ", "KO", "MCD", "MRK", "NKE", "PFE",
    "CRM", "IBM", "INTU", "LOW", "MDT", "MMM", "RTX", "TMO", "UNP", "ZTS"
]
data = yf.download(tickers, start="2022-01-01", end="2025-05-31")["Close"]
returns = data.pct_change().dropna()

# 2. Compute parameters
mu_array = returns.mean().values
cov_matrix = returns.cov().values
n_assets = len(mu_array)
assets = [f"asset_{i+1}" for i in range(n_assets)]

# 3. GANSPy container
m = Container()
i = Set(m, name="i", records=assets)
ii = Alias(m, name="ii", alias_with=i)

# Parameters for transaction cost
x_prev_array = np.array([1/n_assets] * n_assets)  # equally weighted previous portfolio
c_array = np.random.uniform(0.001, 0.01, size=n_assets)  # transaction cost coefficients

x_prev = Parameter(m, name="x_prev", domain=[i],
                   records=[(assets[k], x_prev_array[k]) for k in range(n_assets)])
c = Parameter(m, name="c", domain=[i],
              records=[(assets[k], c_array[k]) for k in range(n_assets)])

# 4. Parameters
Q = Parameter(m, name="Q", domain=[i, ii],
    records=[(assets[r], assets[c], cov_matrix[r, c])
             for r in range(n_assets) for c in range(n_assets)])

mu = Parameter(m, name="mu", domain=[i],
    records=[(assets[k], mu_array[k]) for k in range(n_assets)])

# 5. Decision variables
x = Variable(m, name="x", domain=[i], type="Free")
x.lo[i] = 0
y = Variable(m, name="y", domain=[i], type="Binary")
z = Variable(m, name="z", type="Free")  # uncertainty bound
t = Variable(m, name="t", type="Free")  # worst-case return

# 6. Constraints
budget = Equation(m, name="budget")
budget[...] = Sum(i, x[i]) == 1

card = Equation(m, name="cardinality")
card[...] = Sum(i, y[i]) <= 5

link = Equation(m, name="link", domain=[i])
link[i] = x[i] <= y[i]

risk = Equation(m, name="risk")
risk[...] = Sum([i, ii], Q[i, ii] * x[i] * x[ii]) <= 0.002

robust1 = Equation(m, name="robust_return_ub")
robust1[...] = t + 0.005 * z <= Sum(i, mu[i] * x[i])

robust2 = Equation(m, name="robust_return_soc_1")
robust2[...] = Sum([i, ii], mu[i] * mu[ii] * x[i] * x[ii]) <= z

robust3 = Equation(m, name="robust_return_soc_2")
robust3[...] = -Sum([i, ii], mu[i] * mu[ii] * x[i] * x[ii]) <= z

# 7. Objective
transaction_cost = Sum(i, c[i] * (x[i] - x_prev[i]) * (x[i] - x_prev[i]))
objective_expr = t - transaction_cost

model = Model(m, name="RPOPwCaTC_5_20",
              equations=m.getEquations(),
              problem="MIQCP",
              sense=Sense.MAX,
              objective=objective_expr)

# 8. Solve with SHOT
import sys
model.solve(output=sys.stdout, solver="SHOT", solver_options={"Dual.MIP.Solver": 2})


In [ ]:
# Convert GAMSPy model to GAMS (Note: One needs to download both the .gms and .gdx file)
model.toGams(path="gams")

After this, the files were opened and combined in GAMS Studio by GAMS Convert. Below is the rest of the codes for this type of problem.


## RPOPwCaTC5\_50

In [ ]:
import yfinance as yf
import numpy as np
from gamspy import Container, Set, Parameter, Variable, Equation, Model, Sum, Sense, Alias

# 1. Load market data
tickers = [
    "MMC", "ADP", "VRTX", "CB", "ELV", "REGN", "PNC", "FI", "C", "ADI",
    "CSX", "MDLZ", "SCHW", "PGR", "MU", "DUK", "EQIX", "SO", "HCA",
    "NKE", "PSA", "EMR", "ETN", "ROP", "AIG",
    "MAR", "USB", "D", "EW", "IDXX", "AEP", "FTNT", "ADSK",
    "ROST", "MNST", "MET", "CTAS", "AZO",
    "SPG", "WTW", "CME", "DLR", "PRU", "MSI",
    "ALL", "WELL", "TEL", "SRE", "HLT", "HSY"
]
data = yf.download(tickers, start="2022-01-01", end="2025-05-31")["Close"]
returns = data.pct_change().dropna()

# 2. Compute parameters
mu_array = returns.mean().values
cov_matrix = returns.cov().values
n_assets = len(mu_array)
assets = [f"asset_{i+1}" for i in range(n_assets)]

# 3. GANSPy container
m = Container()
i = Set(m, name="i", records=assets)
ii = Alias(m, name="ii", alias_with=i)

# Parameters for transaction cost
x_prev_array = np.array([1/n_assets] * n_assets)  # equally weighted previous portfolio
c_array = np.random.uniform(0.001, 0.01, size=n_assets)  # transaction cost coefficients

x_prev = Parameter(m, name="x_prev", domain=[i],
                   records=[(assets[k], x_prev_array[k]) for k in range(n_assets)])
c = Parameter(m, name="c", domain=[i],
              records=[(assets[k], c_array[k]) for k in range(n_assets)])

# 4. Parameters
Q = Parameter(m, name="Q", domain=[i, ii],
    records=[(assets[r], assets[c], cov_matrix[r, c])
             for r in range(n_assets) for c in range(n_assets)])

mu = Parameter(m, name="mu", domain=[i],
    records=[(assets[k], mu_array[k]) for k in range(n_assets)])

# 5. Decision variables
x = Variable(m, name="x", domain=[i], type="Free")
x.lo[i] = 0
x.up[i] = 1
y = Variable(m, name="y", domain=[i], type="Binary")
z = Variable(m, name="z", type="Free")  # uncertainty bound
t = Variable(m, name="t", type="Free")  # worst-case return

# 6. Constraints
budget = Equation(m, name="budget")
budget[...] = Sum(i, x[i]) == 1

card = Equation(m, name="cardinality")
card[...] = Sum(i, y[i]) <= 5

link = Equation(m, name="link", domain=[i])
link[i] = x[i] <= y[i]

risk = Equation(m, name="risk")
risk[...] = Sum([i, ii], Q[i, ii] * x[i] * x[ii]) <= 0.002

robust1 = Equation(m, name="robust_return_ub")
robust1[...] = t + 0.005 * z <= Sum(i, mu[i] * x[i])

robust2 = Equation(m, name="robust_return_soc_1")
robust2[...] = Sum([i, ii], mu[i] * mu[ii] * x[i] * x[ii]) <= z

robust3 = Equation(m, name="robust_return_soc_2")
robust3[...] = -Sum([i, ii], mu[i] * mu[ii] * x[i] * x[ii]) <= z

# 7. Objective
transaction_cost = Sum(i, c[i] * (x[i] - x_prev[i]) * (x[i] - x_prev[i]))
objective_expr = t - transaction_cost

model = Model(m, name="RBPOPwCaTC_5_50",
              equations=m.getEquations(),
              problem="MIQCP",
              sense=Sense.MAX,
              objective=objective_expr)

# 8. Solve with SHOT
import sys
model.solve(output=sys.stdout, solver="SHOT", solver_options={"Dual.MIP.Solver": 2, "Model.Convexity.AssumeConvex": True}) # To solve, it is needed to assume that the problem is convex. I don't know why since we basically just add variables from the first version.


## RPOPwCaTC5\_100

In [ ]:
import yfinance as yf
import numpy as np
from gamspy import Container, Set, Parameter, Variable, Equation, Model, Sum, Sense, Alias

# 1. Load market data
tickers = [
    "MMC", "ADP", "VRTX", "CB", "ELV", "REGN", "PNC", "FI", "C", "ADI",
    "CSX", "MDLZ", "SCHW", "PGR", "MU", "DUK", "EQIX", "SO", "HCA",
    "NKE", "PSA", "EMR", "ETN", "ROP", "AIG",
    "MAR", "USB", "D", "EW", "IDXX", "AEP", "FTNT", "ADSK",
    "ROST", "MNST", "MET", "CTAS", "AZO",
    "SPG", "WTW", "CME", "DLR", "PRU", "MSI",
    "ALL", "WELL", "TEL", "SRE", "HLT", "HSY",
    "RMD", "PCG", "DOV", "CMS", "EXC", "DLTR",
    "ATO", "CDW", "HIG", "LDOS", "ALGN", "CNP",
    "VTR", "CRL", "DTE",
    "SJM", "FE", "WAT", "TTWO", "EXR", "BBY", "HOLX", "SWKS", "JKHY", "LKQ",
    "FMC", "TER", "NRG", "AES", "K", "TYL", "PHM", "JBHT",
    "RL", "BR", "ZBRA", "BXP", "GNRC", "VMC", "STE", "ESS", "APA",
    "INCY", "TECH", "PKG", "ADBE", "CMCSA", "PEP", "UNH", "LLY"

]
data = yf.download(tickers, start="2022-01-01", end="2025-05-31")["Close"]
returns = data.pct_change().dropna()

# 2. Compute parameters
mu_array = returns.mean().values
cov_matrix = returns.cov().values
n_assets = len(mu_array)
assets = [f"asset_{i+1}" for i in range(n_assets)]

# 3. GANSPy container
m = Container()
i = Set(m, name="i", records=assets)
ii = Alias(m, name="ii", alias_with=i)

# Parameters for transaction cost
x_prev_array = np.array([1/n_assets] * n_assets)  # equally weighted previous portfolio
c_array = np.random.uniform(0.001, 0.01, size=n_assets)  # transaction cost coefficients

x_prev = Parameter(m, name="x_prev", domain=[i],
                   records=[(assets[k], x_prev_array[k]) for k in range(n_assets)])
c = Parameter(m, name="c", domain=[i],
              records=[(assets[k], c_array[k]) for k in range(n_assets)])

# 4. Parameters
Q = Parameter(m, name="Q", domain=[i, ii],
    records=[(assets[r], assets[c], cov_matrix[r, c])
             for r in range(n_assets) for c in range(n_assets)])

mu = Parameter(m, name="mu", domain=[i],
    records=[(assets[k], mu_array[k]) for k in range(n_assets)])

# 5. Decision variables
x = Variable(m, name="x", domain=[i], type="Free")
x.lo[i] = 0
x.up[i] = 1
y = Variable(m, name="y", domain=[i], type="Binary")
z = Variable(m, name="z", type="Free")  # uncertainty bound
t = Variable(m, name="t", type="Free")  # worst-case return

# 6. Constraints
budget = Equation(m, name="budget")
budget[...] = Sum(i, x[i]) == 1

card = Equation(m, name="cardinality")
card[...] = Sum(i, y[i]) <= 5

link = Equation(m, name="link", domain=[i])
link[i] = x[i] <= y[i]

risk = Equation(m, name="risk")
risk[...] = Sum([i, ii], Q[i, ii] * x[i] * x[ii]) <= 0.002

robust1 = Equation(m, name="robust_return_ub")
robust1[...] = t + 0.005 * z <= Sum(i, mu[i] * x[i])

robust2 = Equation(m, name="robust_return_soc_1")
robust2[...] = Sum([i, ii], mu[i] * mu[ii] * x[i] * x[ii]) <= z

robust3 = Equation(m, name="robust_return_soc_2")
robust3[...] = -Sum([i, ii], mu[i] * mu[ii] * x[i] * x[ii]) <= z

# 7. Objective
transaction_cost = Sum(i, c[i] * (x[i] - x_prev[i]) * (x[i] - x_prev[i]))
objective_expr = t - transaction_cost

model = Model(m, name="RBPOPwCaTC_5_100",
              equations=m.getEquations(),
              problem="MIQCP",
              sense=Sense.MAX,
              objective=objective_expr)

# 8. Solve with SHOT
import sys
model.solve(output=sys.stdout, solver="SHOT", solver_options={"Dual.MIP.Solver": 2, "Model.Convexity.AssumeConvex": True})


## RPOPwCaTC5\_319


In [ ]:
import yfinance as yf
import numpy as np
from gamspy import Container, Set, Parameter, Variable, Equation, Model, Sum, Sense, Alias

# 1. Load market data
tickers = [
     "XOM", "GS", "JPM", "UNH", "HD", "ORCL",
    "CRM", "IBM", "INTU", "LOW", "MDT", "MMM", "RTX", "TMO", "UNP", "ZTS",
    "AMGN", "BLK", "BKNG", "COST", "DE", "DHR", "EL", "FDX", "GE", "GM",
    "HON", "IBM", "ISRG", "JNJ", "LMT", "LIN", "LRCX", "MAR", "MDLZ", "MET",
    "MO", "MS", "NEE", "NOC", "NVDA", "PYPL", "SBUX", "SCHW", "SO", "SPGI",
    "TGT", "TMUS", "TXN", "UPS", "USB", "V", "VLO", "VZ", "WBA", "WFC",
    "XEL", "ZBRA", "ZBH", "SNAP", "DOCU", "UBER", "ZM", "SHOP",
     "ADP", "BMY", "C",  "DOW",  "EMR",  "F",
    "GM",  "HCA",   "LVS",  "PYPL", "BK",
    "AAPL", "MSFT", "AMZN", "NVDA", "GOOGL", "META", "AVGO", "JPM", "TSLA",
    "XOM", "UNH", "LLY", "JNJ", "PG", "MA", "HD", "MRK", "CVX",
    "PEP", "ABBV", "COST", "BAC", "ADBE", "KO", "WMT", "CRM", "NFLX", "TMO",
    "ABT", "ACN", "PFE", "CSCO", "MCD", "DHR", "LIN", "INTC", "QCOM", "NEE",
    "WFC", "TXN", "PM", "MS", "BMY", "AMGN", "HON", "ORCL", "UNP", "AVB",
    "LOW", "UPS", "RTX", "AMD", "GS", "SPGI", "CVS", "SBUX", "BLK", "MDT",
    "ISRG", "LMT", "INTU", "TGT", "BKNG", "GILD",
    "MMC", "ADP", "VRTX", "CB", "ELV", "REGN", "PNC", "FI", "C", "ADI",
    "CSX", "MDLZ", "SCHW", "PGR", "MU", "DUK", "EQIX", "SO", "HCA",
    "BDX", "ICE", "CL", "ITW", "APD", "SHW", "EOG", "TRV", "FDX", "HUM",
    "NKE", "PSA", "EMR", "ETN", "TFC", "GM", "AON", "WM", "ROP", "AIG",
    "MAR", "USB", "MCO", "D", "EW", "IDXX", "AEP", "FTNT", "ADSK",
    "ROST", "MNST", "KDP", "CMG", "MET", "CTAS", "ORLY", "PCAR", "CTSH", "AZO",
    "SPG", "WTW", "CME", "DLR", "PH", "F", "VLO", "PRU", "OXY", "MSI",
    "ALL", "WELL", "TEL", "SRE", "HLT", "HSY", "PEG",
    "EBAY", "DHI", "HES", "MCK", "COF", "KR", "WBD", "ED",
    "DVN", "SBAC", "OTIS", "EFX", "CHTR", "BKR", "DFS", "KEYS", "WMB", "VRSK",
    "FTV", "STT", "PPG", "CEG",  "CTVA",
    "PAYX", "BIIB", "XEL", "LHX", "MTD", "DRI", "ZBH", "KLAC", "TSCO", "TDG",
    "RMD", "PCG", "DOV", "VICI", "CBRE", "GWW", "CMS", "EXC", "DLTR",
    "ATO", "CDW", "HIG", "LDOS", "CNP", "MLM", "LRCX", "AEE", "EIX",
    "RCL", "FANG", "PPL", "FAST", "CARR", "IFF", "COR", "ETR", "TRGP", "MAA",
    "AKAM", "TSN", "EPAM", "NUE", "WEC", "HPE", "IR", "AIZ", "MKC", "RSG",
    "CAH", "VTR", "CHD", "BALL", "NDAQ", "NI", "AVY", "CRL", "DTE",
    "SJM", "FE", "WAT", "TTWO", "EXR", "BBY", "HOLX", "SWKS", "JKHY", "LKQ",
    "FMC", "TER", "NRG", "TYL", "PHM", "JBHT",
    "RL", "BR", "ZBRA", "BXP", "GNRC", "VMC", "STE", "ESS", "APA",
    "INCY", "TECH", "PKG", "CF", "HBAN", "AFL", "NVR", "ALB", "TXT", "DXCM",
    "KIM", "UHS", "BWA", "MOS", "BEN", "IP", "NDSN", "HII", "ARE", "PWR",
    "WHR", "LNT", "EMN", "TAP", "AMCR", "XRAY", "KMX", "L", "CINF", "SNA",
    "FRT", "NOV", "CAG", "GRMN", "WRB", "DXC", "OGN", "TPR", "PNW",
    "A", "SEE", "GL", "HRL", "HAS", "CHRW", "REG", "PARA", "ALLE", "AOS",
    "LKQ", "IPG", "QRVO", "WY", "GNW", "TXT", "PFG"
]
data = yf.download(tickers, start="2022-01-01", end="2025-05-31")["Close"]
returns = data.pct_change().dropna()

# 2. Compute parameters
mu_array = returns.mean().values
cov_matrix = returns.cov().values
n_assets = len(mu_array)
assets = [f"asset_{i+1}" for i in range(n_assets)]

# 3. GANSPy container
m = Container()
i = Set(m, name="i", records=assets)
ii = Alias(m, name="ii", alias_with=i)

# Parameters for transaction cost
x_prev_array = np.array([1/n_assets] * n_assets)  # equally weighted previous portfolio
c_array = np.random.uniform(0.001, 0.01, size=n_assets)  # transaction cost coefficients

x_prev = Parameter(m, name="x_prev", domain=[i],
                   records=[(assets[k], x_prev_array[k]) for k in range(n_assets)])
c = Parameter(m, name="c", domain=[i],
              records=[(assets[k], c_array[k]) for k in range(n_assets)])

# 4. Parameters
Q = Parameter(m, name="Q", domain=[i, ii],
    records=[(assets[r], assets[c], cov_matrix[r, c])
             for r in range(n_assets) for c in range(n_assets)])

mu = Parameter(m, name="mu", domain=[i],
    records=[(assets[k], mu_array[k]) for k in range(n_assets)])

# 5. Decision variables
x = Variable(m, name="x", domain=[i], type="Free")
x.lo[i] = 0
x.up[i] = 1
y = Variable(m, name="y", domain=[i], type="Binary")
z = Variable(m, name="z", type="Free")  # uncertainty bound
t = Variable(m, name="t", type="Free")  # worst-case return

# 6. Constraints
budget = Equation(m, name="budget")
budget[...] = Sum(i, x[i]) == 1

card = Equation(m, name="cardinality")
card[...] = Sum(i, y[i]) <= 5

link = Equation(m, name="link", domain=[i])
link[i] = x[i] <= y[i]

risk = Equation(m, name="risk")
risk[...] = Sum([i, ii], Q[i, ii] * x[i] * x[ii]) <= 0.002

robust1 = Equation(m, name="robust_return_ub")
robust1[...] = t + 0.005 * z <= Sum(i, mu[i] * x[i])

robust2 = Equation(m, name="robust_return_soc_1")
robust2[...] = Sum([i, ii], mu[i] * mu[ii] * x[i] * x[ii]) <= z

robust3 = Equation(m, name="robust_return_soc_2")
robust3[...] = -Sum([i, ii], mu[i] * mu[ii] * x[i] * x[ii]) <= z

# 7. Objective
transaction_cost = Sum(i, c[i] * (x[i] - x_prev[i]) * (x[i] - x_prev[i]))
objective_expr = t - transaction_cost

model = Model(m, name="RBPOPwCaTC_5_319",
              equations=m.getEquations(),
              problem="MIQCP",
              sense=Sense.MAX,
              objective=objective_expr)

# 8. Solve with SHOT
import sys
model.solve(output=sys.stdout, solver="SHOT", solver_options={"Dual.MIP.Solver": 2, "Model.Convexity.AssumeConvex": True})
